In [1]:
! pip install Unidecode

In [2]:
import pandas as pd
import numpy as np
import re
import nltk
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Reshape

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
confusion_matrix, classification_report)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import f1_score, accuracy_score
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.regularizers import l2
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
warnings.filterwarnings('ignore')
from unidecode import unidecode
# Télécharger NLTK data (exécutez une fois)
nltk.download('stopwords')
nltk.download('punkt')
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
import tensorflow as tf
tf.config.optimizer.set_jit(False)  # Disable XLA completely

2025-12-16 17:14:18.272123: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765905258.293442  132986 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765905258.299959  132986 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
!rm -rf /kaggle/working/*

In [4]:
df=pd.read_csv("/kaggle/input/spams-dans-les-commentaires-yt/train_yt.csv")

In [5]:
df.head()

,Unnamed: 0,COMMENT_ID,AUTHOR,DATE,CONTENT,VIDEO_NAME,CLASS
0,1574,LneaDw26bFsFwQBcbaeKWWcSsBB7EA1zRf2nBgT9NyY,Ashim Limbu,NaN,HI IM 14 YEAR RAPPER SUPPORT ME GUY AND CHECK...,Eminem - Love The Way You Lie ft. Rihanna,1
1,859,z13zvnepkwvgirzi104cdf1pyyyecthieew,Ali Altınışık,2015-05-18T10:56:12.254000,Wow dance show﻿,"LMFAO - Party Rock Anthem ft. Lauren Bennett, ...",0
2,651,z12kh3rhbs3duzfea23hzf1qhlyuextzx04,Bobby Carlos,2014-11-06T16:30:30,"Lets be honest, you wouldn't last 1 day on you...",Katy Perry - Roar,0
3,1018,z12cvbyjjmuwvxivx223tpujulrwwdt5j04,ampai gmuer,2015-01-27T13:23:56.061000,Check out this playlist on YouTube:👿👳👳👳👳👳﻿,"LMFAO - Party Rock Anthem ft. Lauren Bennett, ...",1
4,1288,z13xx5w52svttd3lt23kijvwqtvph5m3l,sparkle princess,2015-05-26T07:39:34.920000,"do you guys know, there&#39;s a part two of th...",Eminem - Love The Way You Lie ft. Rihanna,0


In [6]:
df.drop(columns=["Unnamed: 0","COMMENT_ID","DATE"],axis=1,inplace=True)

In [7]:
df.shape

(1369, 4)

In [8]:
df.isnull().sum()

AUTHOR        0
CONTENT       0
VIDEO_NAME    0
CLASS         0
dtype: int64

In [9]:
df["text"]=df["AUTHOR"]+" "+df["CONTENT"]+" "+df["VIDEO_NAME"]

In [10]:
df.drop(columns=["AUTHOR","CONTENT","VIDEO_NAME"],axis=1,inplace=True)

In [11]:
df.head()

,CLASS,text
0,1,Ashim Limbu HI IM 14 YEAR RAPPER SUPPORT ME G...
1,0,Ali Altınışık Wow dance show﻿ LMFAO - Party Ro...
2,0,"Bobby Carlos Lets be honest, you wouldn't last..."
3,1,ampai gmuer Check out this playlist on YouTube...
4,0,"sparkle princess do you guys know, there&#39;s..."


In [12]:
df.isnull().sum()

CLASS    0
text     0
dtype: int64

In [13]:
# def clean_text(text):
#     if pd.isnull(text):
#         return ""
#     text = str(text)
#     text = text.lower()
#     # remove accents
#     text = unidecode(text)
#     # remove urls and mentions
#     text = re.sub(r"http\S+|www\S+", ' ', text)
#     text = re.sub(r"@\w+", ' ', text)
#     # keep letters and numbers
#     text = re.sub(r"[^a-z0-9\s]", ' ', text)
#     # remove digits (optional)
#     text = re.sub(r"\d+", ' ', text)
#     # collapse spaces
#     text = re.sub(r"\s+", ' ', text).strip()
#     return text

# df['text'] = df['text'].astype(str).apply(clean_text)

In [14]:
X = df['text']  # Ou combinez avec features si vous voulez
y = df['CLASS']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train split: {X_train.shape}, Val: {X_val.shape}")

Train split: (1095,), Val: (274,)


In [15]:
# from nltk.corpus import stopwords
# english_stop = stopwords.words('english')
# vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

# X_train_tfidf = vectorizer.fit_transform(X_train)
# X_val_tfidf = vectorizer.transform(X_val)

In [16]:
# lr = LogisticRegression(solver='saga', penalty='l2', C=1.0, max_iter=2000, random_state=SEED)
# lr.fit(X_train_tfidf, y_train)
# y_pred = lr.predict(X_val_tfidf)


In [17]:
# acc = accuracy_score(y_val, y_pred)
# print(f"Accuracy sur Validation: {acc:.4f}")
# print("\nClassification Report:")
# print(classification_report(y_val, y_pred))

In [18]:
test=pd.read_csv("/kaggle/input/spams-dans-les-commentaires-yt/test_yt.csv")
test.drop(columns=["COMMENT_ID","DATE"],axis=1,inplace=True)
test.rename(columns={"Unnamed: 0":"ID"},inplace=True)
Id=test.ID
test.drop(columns=["ID"],axis=1,inplace=True)
test["text"]=test["AUTHOR"]+" "+test["CONTENT"]+" "+test["VIDEO_NAME"]
test.drop(columns=["AUTHOR","CONTENT","VIDEO_NAME"],axis=1,inplace=True)

In [19]:
test.head()

,text
0,Murlock Nightcrawler Charlie from LOST?﻿ Emine...
1,Debora Favacho (Debora Sparkle) BEST SONG EVER...
2,Muhammad Asim Mansha Aslamu Lykum... From Paki...
3,mile panika I absolutely adore watching footba...
4,Sheila Cenabre I really love this video.. http...


In [20]:
# test['text'] = test['text'].astype(str).apply(clean_text)

In [21]:
# preds=vectorizer.transform(test["text"])
# predict=lr.predict(preds)
# submission = pd.DataFrame({'ID': Id, 'CLASS': predict})
# submission.to_csv('submission.csv', index=False)
# print('Saved submission.csv')
# submission.head()

# Model 2

In [22]:
# vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1,2))

# X_train_tfidf = vectorizer.fit_transform(X_train)
# X_val_tfidf = vectorizer.transform(X_val)

In [23]:
# from catboost import CatBoostClassifier
# from sklearn.naive_bayes import MultinomialNB

# mnb=MultinomialNB(alpha=1.0) 
# mnb.fit(X_train_tfidf, y_train)
# y_pred = mnb.predict(X_val_tfidf)

In [24]:
# print("Validation Accuracy:", accuracy_score(y_val, y_pred))
# print(classification_report(y_val, y_pred))

In [25]:
# tf=vectorizer.transform(test["text"])
# predict=mnb.predict(tf)
# submission = pd.DataFrame({'ID': Id, 'CLASS': predict})
# submission.to_csv('mnb_submission.csv', index=False)
# print('Saved submission.csv')
# submission.head()

# Model 3

In [26]:
# vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

# X_train_tfidf = vectorizer.fit_transform(X_train)
# X_val_tfidf = vectorizer.transform(X_val)

In [27]:
# from sklearn.svm import LinearSVC

# svc = LinearSVC()
# svc.fit(X_train_tfidf, y_train)
# y_pred = svc.predict(X_val_tfidf)
# acc = accuracy_score(y_val, y_pred)
# print(f"Accuracy sur Validation: {acc:.4f}")
# print("\nClassification Report:")
# print(classification_report(y_val, y_pred))

In [28]:
# tf=vectorizer.transform(test["text"])
# predict= svc.predict(tf)
# submission = pd.DataFrame({'ID': Id, 'CLASS': predict})
# submission.to_csv('svm_submission.csv', index=False)
# print('Saved submission.csv')
# submission.head()

# Model 4

In [29]:
X = df['text']  
y = df['CLASS']

test = pd.read_csv("/kaggle/input/spams-dans-les-commentaires-yt/test_yt.csv")
test.rename(columns={"Unnamed: 0": "ID"}, inplace=True)
Id = test.ID
test["text"] = test["AUTHOR"] + " " + test["CONTENT"] + " " + test["VIDEO_NAME"]
test_text = test["text"]

In [30]:
vectorizer = TfidfVectorizer(ngram_range=(1,2),max_features=5000) #ngram_range=(1, 2)
X_tfidf = vectorizer.fit_transform(X)
test_tfidf = vectorizer.transform(test_text)

In [31]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []
acc_scores = []
test_preds = np.zeros(len(test))
fold = 1


for train_idx, val_idx in skf.split(X_tfidf, y):
    print(f"Starting Fold {fold}.......................................")
    
    X_train_fold = X_tfidf[train_idx]
    y_train_fold = y.iloc[train_idx].values
    
    X_val_fold = X_tfidf[val_idx]
    y_val_fold = y.iloc[val_idx].values
    
    input_dim = X_train_fold.shape[1]
    
    model = Sequential([
        Dense(128, input_dim=input_dim, activation='relu'),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    
    optimizer = Adam(learning_rate=0.1)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1)
    
    history = model.fit(
        X_train_fold, y_train_fold,
        validation_data=(X_val_fold, y_val_fold),
        epochs=200,
        batch_size=256,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )
    
    y_pred_val = (model.predict(X_val_fold) > 0.5).astype(int).flatten()
    f1 = f1_score(y_val_fold, y_pred_val)
    acc = accuracy_score(y_val_fold, y_pred_val)
    
    print(f"Fold {fold} - F1: {f1:.4f} - Accuracy: {acc:.4f}\n")
    
    f1_scores.append(f1)
    acc_scores.append(acc)
    
    fold_test_preds = model.predict(test_tfidf).flatten()
    test_preds += fold_test_preds / skf.n_splits
    
    fold += 1

Starting Fold 1.......................................


I0000 00:00:1765905262.010798  132986 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/200


I0000 00:00:1765905264.472160  133041 service.cc:148] XLA service 0x7c7d2c005100 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1765905264.472197  133041 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1765905264.653591  133041 cuda_dnn.cc:529] Loaded cuDNN version 90300


1/5 ━━━━━━━━━━━━━━━━━━━━ 11s 3s/step - accuracy: 0.4648 - loss: 0.6941

I0000 00:00:1765905265.909569  133041 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


5/5 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - accuracy: 0.4903 - loss: 1.1322 - val_accuracy: 0.5876 - val_loss: 0.6043 - learning_rate: 0.1000
Epoch 2/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.7488 - loss: 0.5160 - val_accuracy: 0.9380 - val_loss: 0.1935 - learning_rate: 0.1000
Epoch 3/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.9398 - loss: 0.1877 - val_accuracy: 0.9416 - val_loss: 0.1755 - learning_rate: 0.1000
Epoch 4/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.9844 - loss: 0.0764 - val_accuracy: 0.9416 - val_loss: 0.3199 - learning_rate: 0.1000
Epoch 5/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.9822 - loss: 0.0365 - val_accuracy: 0.9635 - val_loss: 0.3179 - learning_rate: 0.1000
Epoch 6/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9924 - loss: 0.0375
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.020000000298023225.
5/5 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.9926 - loss: 0.0371 - val_accuracy: 0.9453 - val_loss: 0.4072 - l

In [32]:
print("Cross-Validation Results")
print(f"Mean F1 Score: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"Mean Accuracy: {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")

Cross-Validation Results
Mean F1 Score: 0.9424 ± 0.0146
Mean Accuracy: 0.9423 ± 0.0127


In [33]:
final_predict = (test_preds >= 0.5).astype(int)
submission = pd.DataFrame({'ID': Id, 'CLASS': final_predict})
submission.to_csv('neu_submission.csv', index=False)
print('Saved neu_submission.csv')
submission.head()

Saved neu_submission.csv


,ID,CLASS
0,1360,0
1,1703,0
2,1146,0
3,1758,1
4,374,1
